# Analyze TI — Exception Case
Evaluate trained model by pair and by symbolic distance.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from collections import defaultdict

from TransitiveTestDataset import TransitiveTestDataset
from TransitiveTrainDataset_Exp import TransitiveTrainDataset_Exp

## Parameters

In [ ]:
MODEL_PATH = "ti_exp_cnn.pt"
N_ITEMS = 8
SAMPLES_PER_PAIR = 2000
DATA_SEED = 42
BATCH_SIZE = 1000

EXCEPTION_WINNER = 5
EXCEPTION_LOSER = 2
EXCEPTION_PAIR = (EXCEPTION_WINNER, EXCEPTION_LOSER)

print(f"Exception pair: {EXCEPTION_WINNER} beats {EXCEPTION_LOSER}")

## Model

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(19968, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

## Load Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = Net().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"Loaded model from {MODEL_PATH}")

## Load Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_train = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST('./data', train=False, transform=transform)

train_dataset = TransitiveTrainDataset_Exp(mnist_train, n=N_ITEMS, samples_per_pair=SAMPLES_PER_PAIR,
                                           seed=DATA_SEED, exception_pair=EXCEPTION_PAIR)
test_dataset = TransitiveTestDataset(mnist_test, n=N_ITEMS, samples_per_pair=SAMPLES_PER_PAIR, seed=DATA_SEED)

print(f"Train pairs: {train_dataset.pairs}")
print(f"Test pairs: {test_dataset.pairs}")

## Evaluation Functions

In [ ]:
def evaluate_by_pair(model, device, dataset, batch_size=1000, exception_pair=None):
    model.eval()
    correct_by_pair = defaultdict(int)
    total_by_pair = defaultdict(int)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sample_idx = 0

    with torch.no_grad():
        for stimuli, labels in loader:
            stimuli, labels = stimuli.to(device), labels.to(device)
            output = model(stimuli)
            preds = output.argmax(dim=1)

            for j in range(stimuli.size(0)):
                i = sample_idx + j
                sample_i = i // 2
                winner_digit, loser_digit, _, _ = dataset.samples[sample_i]

                total_by_pair[(winner_digit, loser_digit)] += 1
                if preds[j].item() == labels[j].item():
                    correct_by_pair[(winner_digit, loser_digit)] += 1

            sample_idx += stimuli.size(0)

    print("Pair (winner > loser) | Accuracy | Correct/Total")
    print("-" * 55)
    for pair in sorted(total_by_pair.keys()):
        acc = correct_by_pair[pair] / total_by_pair[pair]
        marker = "  <-- EXC" if exception_pair and pair == exception_pair else ""
        print(f"  {pair[0]} > {pair[1]}              | {acc:.4f}   | {correct_by_pair[pair]}/{total_by_pair[pair]}{marker}")

    total_correct = sum(correct_by_pair.values())
    total = sum(total_by_pair.values())
    print(f"\n  Overall: {total_correct}/{total} ({100 * total_correct / total:.1f}%)\n")


def evaluate_by_distance(model, device, dataset, batch_size=1000):
    model.eval()
    correct_by_distance = defaultdict(int)
    total_by_distance = defaultdict(int)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sample_idx = 0

    with torch.no_grad():
        for stimuli, labels in loader:
            stimuli, labels = stimuli.to(device), labels.to(device)
            output = model(stimuli)
            preds = output.argmax(dim=1)

            for j in range(stimuli.size(0)):
                i = sample_idx + j
                sample_i = i // 2
                winner_digit, loser_digit, _, _ = dataset.samples[sample_i]
                distance = abs(loser_digit - winner_digit)

                total_by_distance[distance] += 1
                if preds[j].item() == labels[j].item():
                    correct_by_distance[distance] += 1

            sample_idx += stimuli.size(0)

    print("Distance | Accuracy | Correct/Total")
    print("-" * 45)
    for d in sorted(total_by_distance.keys()):
        acc = correct_by_distance[d] / total_by_distance[d]
        print(f"    {d}    | {acc:.4f}   | {correct_by_distance[d]}/{total_by_distance[d]}")

    total_correct = sum(correct_by_distance.values())
    total = sum(total_by_distance.values())
    print(f"\n  Overall: {total_correct}/{total} ({100 * total_correct / total:.1f}%)\n")

## Test Set — By Distance

In [ ]:
print("=== Test Set (non-adjacent pairs) by distance ===")
evaluate_by_distance(model, device, test_dataset, batch_size=BATCH_SIZE)

## Test Set — By Pair

In [ ]:
print("=== Test Set (non-adjacent pairs) by pair ===")
evaluate_by_pair(model, device, test_dataset, batch_size=BATCH_SIZE, exception_pair=EXCEPTION_PAIR)

## Training Set — By Pair

In [ ]:
print("=== Training Set (adjacent + exception) by pair ===")
evaluate_by_pair(model, device, train_dataset, batch_size=BATCH_SIZE, exception_pair=EXCEPTION_PAIR)